# 1. 단어 사전 기반 매핑 복원

## 1) Import

In [ ]:
!pip install triton
!pip install transformers==4.41.2
!pip install accelerate==0.28.0 ### 0.31.0에서 LoRa때문에 버전 내림
!pip install -U bitsandbytes
!pip install peft==0.10.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 85.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 84.7 MB/s eta 0:00:00
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.1
    Uninstalling tokenizers-0.22.1:
      Successfully uninstalled tokenizers-0.22.1
  Attempting uninstall: transformers
    Found existing installation: transformers 4.57.1
    Uninstalling transformers-4.57.1:
      Successfully uninstalled transformers-4.57.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 290.1/290.1 kB 8.7 MB/s eta 0:00:00
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.11.0
    Uninstalling accelerate-1.11.0:
      Successfully uninstalled accelerate-1.11.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.1/199.1 kB 7.1 MB/s eta 0:00:00


In [ ]:
import pandas as pd
from tqdm import tqdm
import os
from google.colab import drive

## 2) Data Load

In [ ]:
drive.mount('/content/drive')
base_path = "/content/drive/MyDrive/data"
data_path = os.path.join(base_path, 'miniproj/')

In [ ]:
train = pd.read_csv(data_path+'train.csv', encoding = 'utf-8-sig')
test = pd.read_csv(data_path+'test.csv', encoding = 'utf-8-sig')

In [ ]:
#train = pd.read_csv('./train.csv', encoding = 'utf-8-sig')
#test = pd.read_csv('./test.csv', encoding = 'utf-8-sig')

## 3) 단어 사전 생성

In [ ]:
match_dict = {}

for input_text, output_text in zip(train['input'], train['output']):
    input_words = input_text.split()
    output_words = output_text.split()
    for iw, ow in zip(input_words, output_words):
        match_dict[iw] = ow

## 4) 변환 적용

In [ ]:
def replace_words(input_text, match_dict):
    words = input_text.split()
    replaced_words = [match_dict.get(word, word) for word in words]
    return " ".join(replaced_words)

In [ ]:
converted_reviews = test['input'].apply(lambda x: replace_words(x, match_dict)).tolist()

## 5) Submission

In [ ]:
submission = pd.read_csv(data_path+'sample_submission.csv', encoding = 'utf-8-sig')

In [ ]:
#submission = pd.read_csv('./sample_submission.csv', encoding = 'utf-8-sig')

In [ ]:
submission['output'] = converted_reviews

In [ ]:
submission.to_csv(data_path+'baseline_submission.csv', index = False, encoding = 'utf-8-sig')

In [ ]:
#submission.to_csv('./baseline_submission.csv', index = False, encoding = 'utf-8-sig')

# 2. LLM 활용 (Gemma)

## 1) Import

In [ ]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline, Trainer, TrainingArguments, DataCollatorForLanguageModeling
from accelerate import Accelerator
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model
from datasets import load_dataset

## 2) Data Load

In [ ]:
train = pd.read_csv(data_path+'train.csv', encoding = 'utf-8-sig')
test = pd.read_csv(data_path+'test.csv', encoding = 'utf-8-sig')

In [ ]:
#train = pd.read_csv('./train.csv', encoding = 'utf-8-sig')
#test = pd.read_csv('./test.csv', encoding = 'utf-8-sig')

In [ ]:
samples = []

for i in range(10):
    sample = f"input : {train['input'][i]} \n output : {train['output'][i]}"
    samples.append(sample)

## 3) Model Load

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type= 'nf4',
    bnb_4bit_use_double_quant = True,
    bnb_4bit_compute_dtype=torch.bfloat16
)
model_id = 'beomi/gemma-ko-2b'
#model_id = 'beomi/gemma-ko-7b'
model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config = bnb_config, device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.95G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/67.1M [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/555 [00:00<?, ?B/s]

3.5) LoRA  

In [ ]:
model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)

In [ ]:
def print_trainable_parameters(model):
    """
    Prints the number of trainable parameters in the model.
    """
    trainable_params = 0
    all_param = 0
    for _, param in model.named_parameters():
        all_param += param.numel()
        if param.requires_grad:
            trainable_params += param.numel()
    print(
        f"trainable params: {trainable_params} || all params: {all_param} || trainable%: {100 * trainable_params / all_param}"
    )

In [ ]:
print_trainable_parameters(model)

trainable params: 0 || all params: 1515268096 || trainable%: 0.0


In [ ]:
print(model)

GemmaForCausalLM(
  (model): GemmaModel(
    (embed_tokens): Embedding(256000, 2048, padding_idx=0)
    (layers): ModuleList(
      (0-17): 18 x GemmaDecoderLayer(
        (self_attn): GemmaSdpaAttention(
          (q_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear4bit(in_features=2048, out_features=256, bias=False)
          (v_proj): Linear4bit(in_features=2048, out_features=256, bias=False)
          (o_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
          (rotary_emb): GemmaRotaryEmbedding()
        )
        (mlp): GemmaMLP(
          (gate_proj): Linear4bit(in_features=2048, out_features=16384, bias=False)
          (up_proj): Linear4bit(in_features=2048, out_features=16384, bias=False)
          (down_proj): Linear4bit(in_features=16384, out_features=2048, bias=False)
          (act_fn): PytorchGELUTanh()
        )
        (input_layernorm): GemmaRMSNorm()
        (post_attention_layernorm): GemmaRMSNorm()
    

In [ ]:
config = LoraConfig(
    r=16,
    lora_alpha=64,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
    ],
    bias="none",
    lora_dropout=0.05,  # Conventional
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, config)
print_trainable_parameters(model)

trainable params: 2506752 || all params: 1517774848 || trainable%: 0.16515967459226205


In [ ]:
def formatting_func(example):
    prompt = f"""
    You are a helpful assistant specializing in restoring obfuscated Korean reviews.
    Your task is to transform the given obfuscated Korean review into a clear, correct,
    and natural-sounding Korean review that reflects its original meaning.
    Below are examples of obfuscated Korean reviews and their restored forms:\n\n
    Example, {samples}
    Spacing and word length in the output must be restored to the same as in the input.
    Do not provide any description. Print only in Korean.\n\n
    input: {example['input']}\noutput: {example['output']}
    """
    return prompt

In [ ]:
def generate_and_tokenize_prompt(ex, max_length=512):
    prompt = formatting_func(ex)
    full_text = prompt + ex["output"] + tokenizer.eos_token


    tokenized = tokenizer(
        full_text,
        truncation=True,
        max_length=max_length,
        padding="max_length"
    )


    labels = tokenized["input_ids"].copy()


    prompt_ids = tokenizer(
        prompt,
        truncation=True,
        max_length=max_length
    )["input_ids"]
    prompt_len = len(prompt_ids)


    labels[:prompt_len] = [-100] * prompt_len


    tokenized["labels"] = labels

    return tokenized


In [ ]:
dataset = load_dataset("csv", data_files=data_path+'train.csv')
full_dataset = dataset["train"]

split_dataset = full_dataset.train_test_split(test_size=0.2, seed=42)

split_dataset = {
    "train": split_dataset["train"],
    "eval": split_dataset["test"]
}

train_dataset = split_dataset["train"]
eval_dataset = split_dataset["eval"]

Generating train split: 0 examples [00:00, ? examples/s]

In [ ]:
print(train_dataset)

Dataset({
    features: ['ID', 'input', 'output'],
    num_rows: 9010
})


In [ ]:
print(eval_dataset)

Dataset({
    features: ['ID', 'input', 'output'],
    num_rows: 2253
})


In [ ]:
tokenized_train_dataset = train_dataset.map(generate_and_tokenize_prompt)
tokenized_val_dataset = eval_dataset.map(generate_and_tokenize_prompt)

Map:   0%|          | 0/9010 [00:00<?, ? examples/s]

Map:   0%|          | 0/2253 [00:00<?, ? examples/s]

In [ ]:
print(tokenized_train_dataset)

Dataset({
    features: ['ID', 'input', 'output', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 9010
})


In [ ]:
model.gradient_checkpointing_disable()

In [ ]:
torch.cuda.empty_cache()
#torch.cuda.reset_peak_memory_stats()

In [ ]:
project = "lora"
run_name = model_id + "-" + project
output_dir = "./" + run_name

training_args = TrainingArguments(
    output_dir="./lora_test",
    per_device_train_batch_size=2,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=1,
    #num_train_epochs = 1,
    max_steps = 1000,
    learning_rate=2e-5,
    bf16=True,
    logging_steps=1,
    save_strategy="no", 
)

data_collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)

trainer = Trainer(
    model=model,  # LoRA + 4-bit 적용 모델
    args=training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_val_dataset,
    data_collator=data_collator
)

model.config.use_cache = False
trainer.train()

print(f"✅ 학습 완료")

lora_output_dir = "./" + run_name + "_lora"
model.save_pretrained(lora_output_dir)
print(f"✅ LoRA weights 저장완료 ({lora_output_dir})")

max_steps is given, it will override any value given in num_train_epochs


Step,Training Loss
1,4.367400
2,4.367400
3,4.367400
4,4.367400
5,4.367400


KeyboardInterrupt: 

In [ ]:
from peft import PeftModel

model = PeftModel.from_pretrained(
    model,               # 원본 모델
    lora_output_dir,  
    device_map="auto", 
    merge_weights=True   # LoRA merge
)

In [ ]:
pipe = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer
)

restored_reviews = []

for index, row in tqdm(test.head(10).iterrows(), desc="진행도"):
#for index, row in tqdm(test.iterrows(), desc="진행도"):
    query = row['input']

    messages = [
        {
            "role": "system",
            "content": f"""
            Transform the given obfuscated Korean review into a clear, correct, and natural-sounding Korean review that reflects its original meaning. Print only in Korean.
            Example, {samples}
            Spacing and word length in the output must be restored to the same as in the input.
            Do not provide any description. Print only in Korean.\n\n
            """
        },
        {
            "role": "user",
            "content": f"input : {query}, output : "
        },
    ]

    prompt = "\n".join([m["content"] for m in messages]).strip()


    outputs = pipe(
        prompt,
        do_sample=True,
        temperature=0.2,
        top_p=0.9,
        max_new_tokens=len(query),
        eos_token_id=pipe.tokenizer.eos_token_id
    )

    generated_text = outputs[0]['generated_text']
    result = generated_text[len(prompt):].strip()


    restored_reviews.append(result)

The model 'PeftModelForCausalLM' is not supported for text-generation. Supported models are ['BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'ElectraForCausalLM', 'ErnieForCausalLM', 'FalconForCausalLM', 'FuyuForCausalLM', 'GemmaForCausalLM', 'GitForCausalLM', 'GPT2LMHeadModel', 'GPT2LMHeadModel', 'GPTBigCodeForCausalLM', 'GPTNeoForCausalLM', 'GPTNeoXForCausalLM', 'GPTNeoXJapaneseForCausalLM', 'GPTJForCausalLM', 'JambaForCausalLM', 'JetMoeForCausalLM', 'LlamaForCausalLM', 'MambaForCausalLM', 'MarianForCausalLM', 'MBartForCausalLM', 'MegaForCausalLM', 'MegatronBertForCausalLM', 'MistralForCausalLM', 'MixtralForCausalLM', 'MptForCausalLM', 'MusicgenForCausalL

In [ ]:
pd.set_option("display.max_colwidth", None)
display(pd.DataFrame(restored_reviews))
pd.reset_option('all')

,0
0,옭 옭 옭 옭 옭 옭 옭 옭 옭 옭 옭 옭 옭 옭 옭 옭
1,"풀룐투갸 엎코, 좀식또 업읍머, 윌뱐 잎츔민든릿 샤있샤윔엡 위썬 호뗄첨렴 관뤽갉 찰 앉 뙨는 누뀜뮈넬오. 까썽"
2,쉭갼 쉭갼 쉭갼 쉭갼 쉭갼 쉭갼 쉭갼 쉭갼 쉭갼 쉭갼 쉭갼 쉭갼 쉭갼 쉭갼 쉭갼 쉭갼 쉭갼 쉭갼 쉭갼 쉭갼 쉭갼 쉭갼 쉭갼 쉭갼 쉭갼 쉭갼 쉭갼 쉭갼 쉭갼 쉭갼 쉭갼 쉭갼 쉭갼 쉭갼 쉭갼 쉭갼 쉭갼 쉭갼 쉭갼 쉭갼 쉭갼 쉭갼 쉭갼 쉭갼 쉭갼 쉭갼 쉭갼 쉭갼 쉭갼 쉭갼 쉭갼 ��
3,"붊 맛있게 먹고 왔습니다. 충간 쏘움광 팔쿄닛갸 잊중짱임 야뉘럇셧 팜몌 퍄톳 쏜륄, 약췸예 깔맸귀윈치 카먀귓윈짐 걔쇽 울엿써 짬울 못 잦여요ㅠ 크련뗄 쀼눈 넒뮤"
4,얾 얾 얾 얾 얾 얾 얾 얾 얾 얾 얾 얾 얾 얾 얾 얾 얾 얾 얾 얾 �
5,"쩨꺄 캑씰돌 넓교 납름 캑꿋했씁닢닦! 쩨꺄 퀵꺄 앉 좋핫써 쩨끄윈핥 떼 웃뿐 헵쁜닝익 위썼눈퉤, 찍퀀뷴잎 찐쩔학계 덱읗햅춤셧숲뉘타! 추챨를 핥 슈 잇단눈 첨운 좋앗쥘만, 쥬챠 관륄읽닢"
6,"잚 뵤없 쑥쇼엔썬 편난학께 쀼룰 캄쌍할 쑤 위써셔 좋앝숩니탸. 캑씰뤼 젼쩨쩍음롯 화잉툰 톤잊럇섦 샤쥐늙 칙움면 덞 홧싸학궤 낮옥골, 잎규옛섶푿떠 쒼밝룰 덞 쒼밝룰 덞 쒼밝룰 덞 쒼밝룰 덞 쒼밝룰 덞 쒼밝룰 덞 쒼밝룰 덞 쒼밝룰 덞 쒼밝룰 덞 쒼밝룰 덞 쒼밝룰 덞 쒼"
7,"졀데 갖지 쥑언님 공공귁콴 윤엉 혼뛔리락교 찌듦위 꽁묾엊뉠략꼬 었찐냐 테독갸 교잤셰엠타 교깩울 짐틀잎 찌빼할련눈 붉찐졀읠 큭칡있교, 씻써른"
8,에어컨 틀어놓고 쉬다가 왔습니다. 에어컨 틀어놓고 쉬다가 왔습니다. 에어컨 틀어놓고 쉬다가 왔습니다. 에어컨 틀어놓고 쉬다가 왔습니다. 에어컨 틀어놓고 쉬다가 왔습니다. 에어컨 틀어놓고 쉬다가 왔습니다. 에어컨 틀어놓고 쉬다가 왔습니다. 에어컨 틀어놓고 쉬다가 왔습니다
9,"쉭툐 쳉거줏씹쿄 썼퓟쓿토 념묶 좋앍옴. 쾅않립는 효텔 1이 췌교잎뉘타ㅠㅠ.\n Example, ['input : 쉭툐 �"


/tmp/ipython-input-2204320661.py:3: FutureWarning: data_manager option is deprecated and will be removed in a future version. Only the BlockManager will be available.
  pd.reset_option('all')
/tmp/ipython-input-2204320661.py:3: FutureWarning: use_inf_as_na option is deprecated and will be removed in a future version. Convert inf values to NaN before operating instead.
  pd.reset_option('all')


## 5) Submission

In [ ]:
submission = pd.read_csv('./sample_submission.csv', encoding = 'utf-8-sig')

In [ ]:
submission['output'] = restored_reviews

In [ ]:
submission.to_csv('./baseline_submission.csv', index = False, encoding = 'utf-8-sig')